In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
import xgboost as xgb


In [2]:
test = pd.read_csv('../../data/CRMLSSold202606.csv')
train = pd.read_csv('../../data/cleaned_df.csv')

/var/folders/5g/sd7vmfvs2rn86tg601yfsjx80000gn/T/ipykernel_37082/1065484362.py:2: DtypeWarning: Columns (0: PoolPrivateYN, 1: FireplaceYN) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../../data/cleaned_df.csv')


In [3]:
train = train[(train['PropertyType'] == 'Residential') & (train['PropertySubType'] == 'SingleFamilyResidence')]
train.head()

,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,ListingContractDate,StateOrProvince,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude
0,1650000.0,3452.0,11255.0,4.0,2.0,2.0,5.0,19,NaN,24059 Regents Park Circle,...,2025-02-09,CA,True,True,True,True,False,3.0,-118.558321,34.404533
1,500000.0,2961.0,8258.0,3.0,2.0,1.0,5.0,25,NaN,11592 Greene Court,...,2025-02-11,CA,False,True,True,True,False,2.0,-117.410510,34.522797
2,1650000.0,2929.0,12907.0,3.0,2.0,1.0,5.0,20,"Tile,Wood",2628 Rudy Street,...,2025-03-03,CA,True,False,True,True,False,3.0,-117.866868,33.968692
3,770000.0,1532.0,3300.0,2.0,2.0,2.0,3.0,17,NaN,10602 Porto Court,...,2024-12-27,CA,False,False,False,True,NaN,2.0,-117.102586,32.825896
4,885000.0,2130.0,8100.0,3.0,1.0,4.0,4.0,8,"Carpet,Vinyl",10420 Oneida Avenue,...,2025-03-03,CA,False,False,False,False,False,2.0,-118.426100,34.260374


In [4]:
test = test[[col for col in train.columns if col in test.columns]]
test = test[(test['PropertyType'] == 'Residential') & (test['PropertySubType'] == 'SingleFamilyResidence')]
test.head()

,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,ListingContractDate,StateOrProvince,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude
2,2250000.0,2629.0,9100.0,3.0,2.0,1.0,4.0,0,NaN,35 Malibu,...,2026-06-30,CA,True,False,True,True,False,3.0,-117.687028,33.512219
3,851000.0,1020.0,5200.0,2.0,NaN,NaN,3.0,0,NaN,2089 PALM BEACH Way,...,2026-06-30,CA,False,NaN,False,True,False,0.0,-121.832491,37.326294
4,950000.0,1568.0,98010.0,2.0,1.0,NaN,3.0,0,NaN,555 Mar Vista Drive,...,2026-05-15,CA,True,False,False,True,NaN,2.0,-117.233819,33.181751
5,245000.0,456.0,11000.0,1.0,1.0,1.0,1.0,0,NaN,24860 4th,...,2026-05-18,CA,False,False,NaN,False,False,0.0,-117.261967,34.106716
6,1175000.0,1704.0,8102.0,2.0,1.0,4.0,4.0,0,NaN,5626 Matilija,...,2026-04-29,CA,False,True,True,True,False,2.0,-118.433103,34.172977


In [5]:
train = train[train['StateOrProvince'] == 'CA']
test = test[test['StateOrProvince'] == 'CA']

In [6]:
def add_features(df):
    model_df = df.copy()
    model_df["LotLivingRatio"] = np.where(df["LotSizeSquareFeet"] > 0, df["LivingArea"] / df["LotSizeSquareFeet"], np.nan)
    model_df['BathroomBedroomRatio'] = np.where(df['BedroomsTotal'] > 0, df['BathroomsTotalInteger']/df['BedroomsTotal'], np.nan)

    amenity_cols = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN']
    amenities = df[amenity_cols].fillna(False).astype(int)
    model_df['TotalAmenityCount'] = amenities.sum(axis=1)

    model_df['FlooringCount'] = df['Flooring'].str.split(',').str.len()
    return model_df

In [7]:
test = add_features(test)
train = add_features(train)

In [8]:
target = 'ClosePrice'

# Categorize columns

cat_col = ['Flooring', 'AssociationFeeFrequency', 
        'MLSAreaMajor', 'ElementarySchool', 'SubdivisionName', 'City', 
        'PurchaseContractDate', 'MiddleOrJuniorSchool', 'HighSchool',
        'HighSchoolDistrict', 'Levels', 'ListingKey', 'CloseDate', 
        'PropertyType', 'ListingKeyNumeric', 'CountyOrParish',
        'PropertySubType', 'ListingId', 'ContractStatusChangeDate',
        'ListingContractDate', 'StateOrProvince', 'UnifiedDistrict']

bool_col = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']

num_col = ['LivingArea', 'LotSizeSquareFeet', 'BathroomsTotalInteger', 
            'Stories', 'MainLevelBedrooms', 'BedroomsTotal', 'DaysOnMarket',
            'LotLivingRatio', 'BathroomBedroomRatio', 'TotalAmenityCount', 'FlooringCount']

required_cols = ['ParkingTotal', 'Longitude', 'Latitude']


# Keep only columns that exist in the dataframe
processed = []
for df in [train, test]:
    num_col_f = [col for col in num_col if col in df.columns]
    cat_col_f = [col for col in cat_col if col in df.columns]
    bool_col_f = [col for col in bool_col if col in df.columns]

    keep_cols = [target] + cat_col_f + bool_col_f + required_cols + num_col_f
    df = df[keep_cols]
    df = df.drop_duplicates()
    processed.append(df)

train, test = processed



In [9]:
import geopandas as gpd

districts = gpd.read_file('../../data/CA_district_areas.geojson')
districts = districts.to_crs("EPSG:4326")
district_info = districts[['DistrictName', 'DistrictType', 'geometry']].copy()

def add_districts(df):
    geo_df = gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']), crs='EPSG:4326')
    df_districts = gpd.sjoin(geo_df, district_info, how="left", predicate="within").drop(columns=['index_right', 'geometry'])
    district_features = (df_districts.reset_index().pivot_table(index='index', columns='DistrictType', values='DistrictName', aggfunc='first'))
    district_split = district_features.rename(columns={'Elementary': 'ElementaryDistrict', 'High': 'HighDistrict', 'Unified': 'UnifiedDistrict'})
    df = df_districts.join(district_split).drop(columns=['DistrictType', 'DistrictName', 'ElementaryDistrict', 'HighDistrict']).drop_duplicates()
    return df



In [10]:
test = add_districts(test)
train = add_districts(train)

In [11]:
def preproc_df(train_df, val_df, test_df):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    # ----------------------------
    # Drop rows with missing values in key columns
    # ----------------------------
    required_cols = ['ParkingTotal', 'Longitude', 'Latitude']
    train_df = train_df.dropna(subset=required_cols)
    val_df = val_df.dropna(subset=required_cols)
    test_df = test_df.dropna(subset=required_cols)

    # Remove invalid values
    train_df = train_df[
        (train_df["ClosePrice"] > 0) &
        (train_df["ClosePrice"] >= 100000) &
        (train_df["LivingArea"] > 0) &
        (train_df["BathroomsTotalInteger"] > 0) &
        (train_df["DaysOnMarket"] > 0)
    ]


    val_df = val_df[
        (val_df["ClosePrice"] > 0) &
        (val_df["ClosePrice"] >= 100000) &
        (val_df["LivingArea"] > 0) &
        (val_df["BathroomsTotalInteger"] > 0) &
        (val_df["DaysOnMarket"] > 0)
    ]

    test_df = test_df[
        (test_df["ClosePrice"] > 0) &
        (test_df["ClosePrice"] >= 100000) &
        (test_df["LivingArea"] > 0) &
        (test_df["BathroomsTotalInteger"] > 0) &
        (test_df["DaysOnMarket"] > 0)
    ]

    # Reset indices to avoid join/alignment bugs during binarization
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    # ----------------------------
    # Missing value handling
    # ----------------------------

    for col in cat_col:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna("Unknown")
        if col in val_df.columns:
            val_df[col] = val_df[col].fillna("Unknown")
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna("Unknown")

    # Missing indicator creation (evaluates missingness per split)
    all_cols = set(train_df.columns) | set(val_df.columns) | set(test_df.columns)
    for col in all_cols:
        has_missing = (
            (col in train_df.columns and train_df[col].isna().sum() > 0) or
            (col in val_df.columns and val_df[col].isna().sum() > 0) or
            (col in test_df.columns and test_df[col].isna().sum() > 0)
        )
        if has_missing:
            if col in train_df.columns: train_df[f"{col}_was_missing"] = train_df[col].isna().astype(int)
            if col in val_df.columns: val_df[f"{col}_was_missing"] = val_df[col].isna().astype(int)
            if col in test_df.columns: test_df[f"{col}_was_missing"] = test_df[col].isna().astype(int)

    # Impute numeric & specific columns using TRAIN statistics
    if "YearBuilt" in train_df.columns:
        year_median = train_df["YearBuilt"].median()
        train_df["YearBuilt"] = train_df["YearBuilt"].fillna(year_median)
        val_df["YearBuilt"] = val_df["YearBuilt"].fillna(year_median)
        test_df["YearBuilt"] = test_df["YearBuilt"].fillna(year_median)

    if "StreetNumberNumeric" in train_df.columns:
        train_df["StreetNumberNumeric"] = train_df["StreetNumberNumeric"].fillna(-1)
        val_df["StreetNumberNumeric"] = val_df["StreetNumberNumeric"].fillna(-1)
        test_df["StreetNumberNumeric"] = test_df["StreetNumberNumeric"].fillna(-1)

    train_df[bool_col] = train_df[bool_col].fillna(False)
    val_df[bool_col] = val_df[bool_col].fillna(False)
    test_df[bool_col] = test_df[bool_col].fillna(False)

    for col in num_col:
        if col in train_df.columns:
            median = train_df[col].median()
            train_df[col] = train_df[col].fillna(median)
            val_df[col] = val_df[col].fillna(median)
            test_df[col] = test_df[col].fillna(median)

    if "AssociationFee" in train_df.columns:
        train_df["AssociationFee"] = train_df["AssociationFee"].fillna(0)
        val_df["AssociationFee"] = val_df["AssociationFee"].fillna(0)
        test_df["AssociationFee"] = test_df["AssociationFee"].fillna(0)

    for col in ["GarageSpaces", "ParkingTotal"]:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna(0)
            val_df[col] = val_df[col].fillna(0)
            test_df[col] = test_df[col].fillna(0)

    # ----------------------------
    # Boolean encoding
    # ----------------------------

    train_df[bool_col] = train_df[bool_col].astype(int)
    val_df[bool_col] = val_df[bool_col].astype(int)
    test_df[bool_col] = test_df[bool_col].astype(int)

    # ----------------------------
    # MultiLabel Encoding
    # ----------------------------

    multi_cols = ["Flooring", "Levels"]

    for col in multi_cols:
        train_df[col] = train_df[col].fillna("").str.split(",")
        val_df[col] = val_df[col].fillna("").str.split(",")
        test_df[col] = test_df[col].fillna("").str.split(",")

        mlb = MultiLabelBinarizer()

        train_encoded = pd.DataFrame(
            mlb.fit_transform(train_df[col]),
            columns=[f"{col}_{c}" for c in mlb.classes_],
            index=train_df.index
        )

        val_encoded = pd.DataFrame(
            mlb.transform(val_df[col]),
            columns=[f"{col}_{c}" for c in mlb.classes_],
            index=val_df.index
        )

        test_encoded = pd.DataFrame(
            mlb.transform(test_df[col]),
            columns=[f"{col}_{c}" for c in mlb.classes_],
            index=test_df.index
        )

        train_df = train_df.drop(columns=col).join(train_encoded)
        val_df = val_df.drop(columns=col).join(val_encoded)
        test_df = test_df.drop(columns=col).join(test_encoded)

    # ----------------------------
    # Ordinal Encoding
    # ----------------------------

    mapping = {
        "Unknown": 0,
        "Monthly": 1,
        "Quarterly": 2,
        "SemiAnnually": 3,
        "Annually": 4
    }

    train_df["AssociationFeeFrequency"] = train_df["AssociationFeeFrequency"].map(mapping)
    val_df["AssociationFeeFrequency"] = val_df["AssociationFeeFrequency"].map(mapping)
    test_df["AssociationFeeFrequency"] = test_df["AssociationFeeFrequency"].map(mapping)

    # ----------------------------
    # One-Hot Encoding
    # ----------------------------

    one_hot = [
        "CountyOrParish",
        "StateOrProvince",
        "City",
        "PropertyType",
        "PropertySubType",
        'UnifiedDistrict'
    ]

    for col in one_hot:
        top = train_df[col].value_counts().head(200).index

        train_df[col] = train_df[col].where(train_df[col].isin(top), "Other")
        val_df[col] = val_df[col].where(val_df[col].isin(top), "Other")
        test_df[col] = test_df[col].where(test_df[col].isin(top), "Other")

    train_df = pd.get_dummies(train_df, columns=one_hot, drop_first=True)
    val_df = pd.get_dummies(val_df, columns=one_hot, drop_first=True)
    test_df = pd.get_dummies(test_df, columns=one_hot, drop_first=True)

    # Align columns across train, val, and test to ensure identical structure
    train_df, val_df = train_df.align(val_df, join="left", axis=1, fill_value=0)
    train_df, test_df = train_df.align(test_df, join="left", axis=1, fill_value=0)

    # ----------------------------
    # Standardization
    # ----------------------------

    scale = [
        "LivingArea",
        "LotSizeSquareFeet",
        "AssociationFee",
        "DaysOnMarket"
    ]

    scale = [c for c in scale if c in train_df.columns]

    scaler = StandardScaler()
    train_df[scale] = scaler.fit_transform(train_df[scale])
    val_df[scale] = scaler.transform(val_df[scale])
    test_df[scale] = scaler.transform(test_df[scale])

    train_df = train_df.drop(columns=cat_col, errors='ignore')
    val_df = val_df.drop(columns=cat_col, errors='ignore')
    test_df = test_df.drop(columns=cat_col, errors='ignore')

    train_df = train_df.drop_duplicates()
    val_df = val_df.drop_duplicates()
    test_df = test_df.drop_duplicates()

    return train_df, val_df, test_df

In [13]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def mdape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.median(np.abs((y_true - y_pred) / y_true)) * 100

In [14]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(train, test_size=0.2, random_state=42)
# 1. Run preprocessing first on your split dataframes
train_proc, val_proc, test_proc = preproc_df(train_df, val_df, test)

# 2. Extract X and y AFTER preprocessing has dropped invalid rows
target_col = "ClosePrice"

X_train = train_proc.drop(columns=[target_col], errors="ignore")
y_train = train_proc[target_col]

X_val = val_proc.drop(columns=[target_col], errors="ignore")
y_val = val_proc[target_col]

X_test = test_proc.drop(columns=[target_col], errors="ignore")
y_test = test_proc[target_col]
# 3. Verify lengths before fitting
print(f"X_train rows: {len(X_train)} | y_train rows: {len(y_train)}") # Should both be 61973
print(f"X_val rows:   {len(X_val)}   | y_val rows:   {len(y_val)}")
print(f"X_test rows:   {len(X_test)} | y_test rows:   {len(y_test)}")


/Users/lydiachin/miniforge3/envs/idx/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['Unknown'] will be ignored
  warnings.warn(


X_train rows: 61893 | y_train rows: 61893
X_val rows:   15472   | y_val rows:   15472
X_test rows:   12254 | y_test rows:   12254


In [ ]:
# Column Alignment Check
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")
print()

train_cols = set(X_train.columns)
val_cols = set(X_val.columns)
test_cols = set(X_test.columns)

train_only = train_cols - val_cols - test_cols
val_only = val_cols - train_cols - test_cols
test_only = test_cols - train_cols - val_cols

in_train_not_test = train_cols - test_cols
in_test_not_train = test_cols - train_cols
in_train_not_val = train_cols - val_cols
in_val_not_train = val_cols - train_cols

print(f"Columns in train but not test ({len(in_train_not_test)}):")
print(sorted(in_train_not_test))
print()
print(f"Columns in test but not train ({len(in_test_not_train)}):")
print(sorted(in_test_not_train))
print()
print(f"Columns in train but not val ({len(in_train_not_val)}):")
print(sorted(in_train_not_val))
print()
print(f"Columns in val but not train ({len(in_val_not_train)}):")
print(sorted(in_val_not_train))
print()


if in_train_not_test:
    print("=== Columns dropped for test (will be zero-filled) ===")
    print(X_train[sorted(in_train_not_test)].describe().T[['mean', 'std', 'min', 'max']])
    print()

# Sanity check: after alignment, are val/test actually fully aligned to train?
print("Fully aligned train==val columns:", list(X_train.columns) == list(X_val.columns))
print("Fully aligned train==test columns:", list(X_train.columns) == list(X_test.columns))
print("Fully aligned val==test columns:  ", list(X_val.columns) == list(X_test.columns))
print()


print("=== ClosePrice floor check ===")
print(f"y_train min: {y_train.min():,.0f}")
print(f"y_val min:   {y_val.min():,.0f}")
print(f"y_test min:  {y_test.min():,.0f}")
print(f"y_test rows below $100,000: {(y_test < 100000).sum()} / {len(y_test)}")
print(f"y_test rows below $10,000:  {(y_test < 10000).sum()} / {len(y_test)}")


X_train shape: (61893, 497)
X_val shape:   (15472, 497)
X_test shape:  (12254, 497)

Columns in train but not test (0):
[]

Columns in test but not train (0):
[]

Columns in train but not val (0):
[]

Columns in val but not train (0):
[]

Fully aligned train==val columns: True
Fully aligned train==test columns: True
Fully aligned val==test columns:   True

=== ClosePrice floor check ===
y_train min: 100,000
y_val min:   102,000
y_test min:  100,000
y_test rows below $100,000: 0 / 12254
y_test rows below $10,000:  0 / 12254


In [ ]:
model_results = []

n_estimators = [300, 500, 700]
max_depth = [4, 6, 8]
learning_rate = [0.01, 0.05, 0.1]

for depth in max_depth:
    for rate in learning_rate:
        for n in n_estimators:
            final = xgb.XGBRegressor(
                n_estimators=n, 
                max_depth=depth, 
                learning_rate=rate, 
                random_state=42
            )
            
            start_time = time.time()
            final.fit(X_train, y_train)
            training_seconds = time.time() - start_time

            # Predictions
            train_preds = final.predict(X_train)
            val_preds = final.predict(X_val)
            
            # Metrics (Matching exact train vs. validation arrays)
            train_r2 = r2_score(y_train, train_preds)
            val_r2 = r2_score(y_val, val_preds)
            rmse = root_mean_squared_error(y_val, val_preds)
            mae = mean_absolute_error(y_val, val_preds)
            
            model_results.append({
                "max_depth": depth,
                "learning_rate": rate,
                "n_estimators": n,
                "train_r2": train_r2,
                "val_r2": val_r2,
                "mae": mae,
                "rmse": rmse,
                "mape": mape(y_val, val_preds),
                "mdape": mdape(y_val, val_preds),
                "time": training_seconds
            })

# View and sort results
comparison = pd.DataFrame(model_results)
comparison.sort_values('val_r2', ascending=False)

,max_depth,learning_rate,n_estimators,train_r2,val_r2,mae,rmse,mape,mdape,time
26,8,0.10,700,0.973261,0.917953,92985.504849,161538.492430,10.315170,6.851473,5.108473
25,8,0.10,500,0.967074,0.916637,94078.595967,162828.664453,10.460234,6.976188,3.849579
23,8,0.05,700,0.960612,0.914415,95643.542849,164985.025391,10.672728,7.064537,5.317922
17,6,0.10,700,0.952923,0.914300,96351.623979,165095.006765,10.815132,7.219826,4.369375
24,8,0.10,300,0.958856,0.913684,96425.178219,165688.087323,10.775140,7.177437,2.584200
22,8,0.05,500,0.954982,0.911672,97555.848854,167607.388384,10.922911,7.220774,4.083788
16,6,0.10,500,0.945177,0.910755,98883.101128,168475.023393,11.139179,7.458850,3.306524
21,8,0.05,300,0.947999,0.907881,100315.515650,171166.659069,11.303169,7.462450,2.791100
14,6,0.05,700,0.937041,0.907302,101383.687618,171704.262442,11.478464,7.727226,4.535908
8,4,0.10,700,0.927227,0.905032,103699.237708,173793.893018,11.795865,8.060707,3.873916


max_depth = 8, learning_rate = 0.1, and n_estimators = 700 result in the model wiht the highest validation R2

In [17]:

X_full = pd.concat([X_train, X_val], axis=0)
y_full = pd.concat([y_train, y_val], axis=0)


# 3. Instantiate model with row 26's best parameters
best_model = xgb.XGBRegressor(
    max_depth=8,
    learning_rate=0.10,
    n_estimators=700,
    random_state=42,
    n_jobs=-1
)


# Fit & predict
best_model.fit(X_full, y_full)
train_preds = best_model.predict(X_full)
test_preds = best_model.predict(X_test)

# Calculate metrics against y_test (not X_test)
train_r2 = r2_score(y_full, train_preds)
test_r2 = r2_score(y_test, test_preds)
rmse = root_mean_squared_error(y_test, test_preds)
mae = mean_absolute_error(y_test, test_preds)

results = {
    "train_r2": train_r2,
    "test_r2": test_r2,
    "mae": mae,
    "rmse": rmse,
    "mape": mape(y_test, test_preds),
    "mdape": mdape(y_test, test_preds),
}

pd.DataFrame([results])

,train_r2,test_r2,mae,rmse,mape,mdape
0,0.970625,0.423088,293526.428959,1.099990e+06,15.805456,9.558325
